# M07.3 — reproduzir o Renewcast no Databricks

**Resultado:** uma tabela Delta de geração e uma tabela Delta de treino, em Unity Catalog; comparação integral com o treino local; recibo com identidades, versões, parâmetros e durações.

Use o **Serverless** do Databricks Free Edition. Execute a célula **1** primeiro, envie os três arquivos ao caminho impresso e só então execute as demais. O notebook já contém toda a transformação. Não instalar PySpark, Java ou JARs; não executar este notebook no PowerShell.

A entrada é o Parquet original, não a tabela com a revisão simulada de M07.2. O objetivo é transportar uma transformação aceita para outro ambiente. O snapshot local e a tabela cloud têm identidades/versionamentos independentes. Fuso físico e disponibilidade histórica continuam hipóteses.

## 1. Catálogo, schema e volume

`catalogo.schema.tabela` identifica uma tabela. `/Volumes/catalogo/schema/volume/...` identifica arquivos de apoio, como os Parquets recebidos. O volume de entrada não é o diretório físico das tabelas gerenciadas.

O widget `catalogo` usa o catálogo atual. Se ele não for seu catálogo gravável, selecione o nome visível em **Catalog** e execute esta célula novamente. O schema `renewcast_m07` isola os objetos desta aula.

In [0]:
import csv
import hashlib
import json
import math
import platform
import re
import time
import uuid
from contextlib import redirect_stdout
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path
from pyspark.sql import functions as F

dbutils.widgets.text("catalogo", spark.sql("SELECT current_catalog()").first()[0], "Catalogo gravavel")
CATALOGO = dbutils.widgets.get("catalogo").strip()
ESQUEMA, VOLUME = "renewcast_m07", "arquivos"
if not re.fullmatch(r"[A-Za-z0-9_-]+", CATALOGO) or CATALOGO in {"hive_metastore", "system", "samples"}:
    raise ValueError("Escolha no widget um catalogo Unity Catalog gravavel do seu workspace.")
NS = f"`{CATALOGO}`.`{ESQUEMA}`"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {NS}").collect()
spark.sql(f"CREATE VOLUME IF NOT EXISTS {NS}.`{VOLUME}`").collect()
ROOT_VOLUME = Path(f"/Volumes/{CATALOGO}/{ESQUEMA}/{VOLUME}")
INPUT_DIR = ROOT_VOLUME / "entrada_m073"
dbutils.fs.mkdirs(str(INPUT_DIR))
spark.conf.set("spark.sql.session.timeZone", "UTC")
EDICAO = "free_edition"  # Altere se estiver usando workspace de outra edicao.
print("Envie geracao.parquet, treino_referencia.parquet e manifesto_entrada.json para:")
print(INPUT_DIR)
print("Depois do upload, execute a partir da celula 2.")

Envie geracao.parquet, treino_referencia.parquet e manifesto_entrada.json para:
/Volumes/workspace/renewcast_m07/arquivos/entrada_m073
Depois do upload, execute a partir da celula 2.


## 2. Entradas identificadas pelo conteúdo

No menu **New → Add or upload data → Upload files to a volume**, envie os três arquivos de `dist/databricks/CJU_BA5EIGT/m07_3` para o caminho acima. Não use a opção de criar uma tabela por upload: faremos a criação explicitamente.

O hash detecta alterações desde a preparação. Ele não certifica a veracidade da fonte. Os arquivos são pequenos; a leitura de bytes abaixo ocorre no Python apenas para conferir a transferência.

In [0]:
def sha_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

for name in ("geracao.parquet", "treino_referencia.parquet", "manifesto_entrada.json"):
    if not (INPUT_DIR / name).is_file():
        raise FileNotFoundError(f"Falta {INPUT_DIR / name}. Conclua o upload antes de executar esta celula.")
manifest = json.loads((INPUT_DIR / "manifesto_entrada.json").read_text(encoding="utf-8"))
if manifest.get("schema") != "renewcast-databricks-input-v1":
    raise ValueError("Manifesto de entrada desconhecido.")
PACKAGE_ID = manifest["package_id"]
if not re.fullmatch(r"[0-9a-f]{64}", PACKAGE_ID):
    raise ValueError("package_id invalido.")
digest_input = {k:v for k,v in manifest.items() if k not in {"package_id", "preparation_sources_sha256"}}
canonical = json.dumps(digest_input,sort_keys=True,separators=(",",":"),ensure_ascii=False).encode()
if hashlib.sha256(canonical).hexdigest() != PACKAGE_ID:
    raise ValueError("Identidade do pacote diverge do seu conteudo.")
if set(manifest["files"]) != {"geracao.parquet", "treino_referencia.parquet"}:
    raise ValueError("Conjunto de arquivos diferente do contrato.")
for name, info in manifest["files"].items():
    path = INPUT_DIR / name
    if sha_file(path) != info["sha256"] or path.stat().st_size != info["bytes"]:
        raise ValueError(f"Arquivo transferido diverge: {name}")
PARAMS = manifest["parameters"]
if PARAMS != {"horizon_hours":24, "availability_delay_hours_assumed":2, "window_hours":6, "cutoff":"2026-07-11T06:00:00"}:
    raise ValueError("Esta reproducao usa o protocolo h24/d2/w6 ja aceito.")
CORTE = datetime.fromisoformat(PARAMS["cutoff"])
IMPL_ID = "m073-temporal-v1"
SOURCE_TABLE = f"{NS}.`geracao_m073_{PACKAGE_ID[:12]}`"
TRAIN_TABLE = f"{NS}.`treino_m073_{PACKAGE_ID[:12]}`"
geracao = spark.read.parquet(str(INPUT_DIR / "geracao.parquet")).select(
    F.col("din_instante").cast("timestamp_ntz").alias("din_instante"),
    F.col("val_geracao").cast("double").alias("val_geracao"))
referencia = spark.read.parquet(str(INPUT_DIR / "treino_referencia.parquet"))
for name in ("forecast_origin", "history_end", "target_time", "target_available_at"):
    referencia = referencia.withColumn(name,F.col(name).cast("timestamp_ntz"))
print("Pacote:", PACKAGE_ID)
print("Entrada conferida; nenhuma tabela escrita ate este ponto.")

Pacote: f66e094498ef5cd3586d167b691027e5e8af312674ba4bb0b59473f0f6f2e656
Entrada conferida; nenhuma tabela escrita ate este ponto.


## 3. Transformação e seleção temporal

Estas são as quatro funções do laboratório aceito: validar, completar a grade, construir as features e selecionar o treino. Mantivemos o contrato e explicitamos o literal do corte como `TIMESTAMP_NTZ`: um datetime Python sem fuso não deve passar por uma conversão implícita que dependa do fuso do processo. O `Window.partitionBy("serie")` mantém cada série junta; dividir por dia quebraria lags e janelas que cruzam a meia-noite.

As expressões retornam planos. `count`, `first`, a gravação e a comparação posterior solicitam execução. Os `take(1)` da validação trazem no máximo uma contraprova, não o ano inteiro.

In [0]:
FEATURES = ("geracao_em_history_end", "media_historico_mwmed")

def validar_entrada(horaria):
    from pyspark.sql import functions as F

    if set(horaria.columns) != {"din_instante", "val_geracao"}:
        raise ValueError("A entrada selecionada deve ter timestamp e geracao.")
    if not horaria.take(1):
        raise ValueError("Serie vazia.")
    invalida = horaria.filter(
        F.col("din_instante").isNull()
        | (F.date_trunc("hour", "din_instante") != F.col("din_instante"))
        | F.isnan("val_geracao") | (F.abs(F.col("val_geracao")) == float("inf"))
    )
    if invalida.take(1):
        raise ValueError("Ha timestamp ausente/fora da hora ou valor NaN/infinito; NULL de geracao e permitido.")
    if horaria.groupBy("din_instante").count().filter("count > 1").take(1):
        raise ValueError("Timestamp duplicado: resolva a origem da duplicidade antes de construir as features.")

def completar_horas(horaria):
    """Uma linha por hora, inclusive quando a observacao nao chegou."""
    from pyspark.sql import functions as F

    limites = horaria.agg(F.min("din_instante").alias("inicio"), F.max("din_instante").alias("fim"))
    grade = limites.select(F.explode(F.sequence("inicio", "fim", F.expr("INTERVAL 1 HOUR"))).alias("din_instante"))
    marcada = horaria.withColumn("registro_presente", F.lit(True))
    return grade.join(marcada, "din_instante", "left").fillna(False, subset=["registro_presente"])

def construir_features(grade, horizonte=24, atraso=2, janela=6):
    """Precondicao: grade horaria unica/completa; timestamps no mesmo eixo.

    Na origem o, observacoes ate o-atraso estao disponiveis (fronteira inclusiva).
    Uma media so e utilizavel quando todas as horas da janela tem valor.
    Alvo e disponibilidade do alvo sao metadados; nao pertencem a FEATURES.
    """
    from pyspark.sql import Window, functions as F

    ordem = Window.partitionBy("serie").orderBy("din_instante")
    historico = ordem.rowsBetween(-(atraso + janela - 1), -atraso)
    tabela = (
        grade.withColumn("serie", F.lit("CJU_BA5EIGT"))
        .withColumn("geracao_em_history_end", F.lag("val_geracao", atraso).over(ordem))
        .withColumn("horas_validas_historico", F.count("val_geracao").over(historico))
        .withColumn("media_parcial", F.avg("val_geracao").over(historico))
        .withColumn("media_historico_mwmed", F.when(F.col("horas_validas_historico") == janela, F.col("media_parcial")))
        .withColumn("target_generation", F.lead("val_geracao", horizonte).over(ordem))
        .withColumn("history_end", F.expr(f"din_instante - INTERVAL {atraso} HOURS"))
        .withColumn("target_time", F.expr(f"din_instante + INTERVAL {horizonte} HOURS"))
        .withColumn("target_available_at", F.expr(f"din_instante + INTERVAL {horizonte + atraso} HOURS"))
        .withColumn("features_validas", F.col("geracao_em_history_end").isNotNull() & F.col("media_historico_mwmed").isNotNull())
        .withColumnRenamed("din_instante", "forecast_origin")
    )
    return tabela.select(
        "serie", "forecast_origin", "history_end", "target_time", "target_available_at",
        "registro_presente", *FEATURES, "horas_validas_historico", "features_validas", "target_generation",
    )

def selecionar_treino(tabela, corte):
    """Implementacao autoral recebida: preserva colunas e seleciona linhas elegiveis.

    Regras: features_validas; target_generation preenchido;
    target_available_at ESTRITAMENTE anterior a corte.
    corte e datetime sem fuso, no mesmo eixo dos rotulos da serie.
    Esta funcao so e chamada quando voce informa --corte-treino.
    """
    from pyspark.sql import functions as F

    return tabela.filter(
        F.col("features_validas")
        & F.col("target_generation").isNotNull()
        & (F.col("target_available_at") < F.lit(corte.isoformat()).cast("timestamp_ntz"))
    )

## 4. Publicação controlada e comparação integral

Uma tabela nova é criada uma vez. Se ela já existe, deve ter o mesmo identificador de pacote e a mesma finalidade; seu conteúdo será conferido. Não há `overwrite`, `DROP` nem `VACUUM` neste notebook.

O recibo captura o **ID da tabela e sua versão**. A versão sozinha é insuficiente: versão 0 de duas tabelas diferentes não identifica os mesmos dados. A comparação faz um `FULL JOIN` pela chave, confere ausências e todas as colunas; para valores double, tolerância absoluta de 1e-9 MWmed.

In [0]:
def publicar_uma_vez(df, table_name, role):
    # table_name e composto somente por identificadores validados na configuracao.
    exists = spark.catalog.tableExists(table_name)
    if exists:
        props = {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES {table_name}").collect()}
        expected = {"renewcast.package_id": PACKAGE_ID, "renewcast.role": role,
                    "renewcast.implementation": IMPL_ID}
        if any(props.get(k) != v for k,v in expected.items()):
            raise ValueError(f"Tabela preexistente nao pertence a esta reproducao: {table_name}")
    else:
        df.createOrReplaceTempView("m073_publicacao_temporaria")
        spark.sql(f"""CREATE TABLE {table_name} USING DELTA
          TBLPROPERTIES ('renewcast.package_id'='{PACKAGE_ID}',
                         'renewcast.role'='{role}',
                         'renewcast.implementation'='{IMPL_ID}')
          AS SELECT * FROM m073_publicacao_temporaria""").collect()
    detail = spark.sql(f"DESCRIBE DETAIL {table_name}").first()
    version = int(spark.sql(f"DESCRIBE HISTORY {table_name} LIMIT 1").first()["version"])
    return {"name":table_name, "table_id":detail["id"], "version":version,
            "reused":exists, "format":detail["format"]}


def comparar_frames(left, right, keys):
    if set(left.columns) != set(right.columns):
        raise ValueError("As tabelas nao possuem as mesmas colunas.")
    if any(left.schema[name].dataType != right.schema[name].dataType for name in left.columns):
        raise ValueError("Os tipos das colunas divergem.")
    for df in (left,right):
        if df.groupBy(*keys).count().filter("count > 1").take(1):
            raise ValueError("Chave duplicada na comparacao.")
    a = left.withColumn("__left_present",F.lit(True)).alias("a")
    b = right.withColumn("__right_present",F.lit(True)).alias("b")
    joined = a.join(b,keys,"full")
    bad = F.col("__left_present").isNull() | F.col("__right_present").isNull()
    maxima=[]
    for name in left.columns:
        if name in keys:
            continue
        av,bv=F.col(f"a.`{name}`"),F.col(f"b.`{name}`")
        if left.schema[name].dataType.simpleString() in {"double","float"}:
            finite=(~F.isnan(av)) & (~F.isnan(bv)) & (F.abs(av)!=float("inf")) & (F.abs(bv)!=float("inf"))
            equal=av.eqNullSafe(bv) | (finite & (F.abs(av-bv)<=1e-9))
            maxima.append(F.max(F.abs(av-bv)).alias(f"max_diff_{name}"))
        else:
            equal=av.eqNullSafe(bv)
        bad=bad | (~F.coalesce(equal,F.lit(False)))
    result=joined.agg(
        F.count("*").alias("rows_union"),
        F.sum(F.col("__left_present").isNotNull().cast("long")).alias("rows_left"),
        F.sum(F.col("__right_present").isNotNull().cast("long")).alias("rows_right"),
        F.sum(bad.cast("long")).alias("mismatched_rows"), *maxima).first().asDict()
    if result["mismatched_rows"] != 0:
        raise ValueError(f"Comparacao falhou: {result}")
    return result


validar_entrada(geracao)
if geracao.count() != manifest["generation_rows"]:
    raise ValueError("Numero de registros de geracao diverge do manifesto.")
source_info = publicar_uma_vez(geracao,SOURCE_TABLE,"generation")
fonte_fixada=spark.read.option("versionAsOf",source_info["version"]).table(SOURCE_TABLE)
source_comparison=comparar_frames(fonte_fixada,geracao,["din_instante"])
print("Fonte fixada:", json.dumps(source_info,ensure_ascii=False))
print("Comparacao integral da fonte:", source_comparison)

Fonte fixada: {"name": "`workspace`.`renewcast_m07`.`geracao_m073_f66e094498ef`", "table_id": "e3c3dc5d-912c-4933-a14e-a7b476bbb374", "version": 0, "reused": false, "format": "delta"}
Comparacao integral da fonte: {'rows_union': 8760, 'rows_left': 8760, 'rows_right': 8760, 'mismatched_rows': 0, 'max_diff_val_geracao': 0.0}


## 5. Duas configurações; resultado obrigatoriamente igual

Este ensaio muda somente `spark.sql.shuffle.partitions`, opção aceita no serverless. Ele mede a duração da **comparação completa**, que inclui leitura, transformação e auditoria, e não um tempo puro de inferência. Uma observação por configuração não permite escolher a mais rápida com confiança.

Há apenas uma série. O valor 4 não significa quatro máquinas e não garante quatro tarefas úteis na janela. Leia o `Exchange` próximo do `Window` em cada plano. O otimizador adaptativo também pode ajustar o plano durante a execução.

In [0]:
runs=[]
plans={}
previous_partitions=spark.conf.get("spark.sql.shuffle.partitions")
try:
    for partitions in (2,4):
        spark.conf.set("spark.sql.shuffle.partitions",str(partitions))
        start=time.perf_counter()
        base=spark.read.option("versionAsOf",source_info["version"]).table(SOURCE_TABLE)
        features=construir_features(completar_horas(base),
            PARAMS["horizon_hours"],PARAMS["availability_delay_hours_assumed"],PARAMS["window_hours"])
        selected=selecionar_treino(features,CORTE)
        comparison=comparar_frames(selected,referencia,["serie","forecast_origin"])
        seconds=time.perf_counter()-start
        if comparison["rows_left"] != manifest["training_rows"]:
            raise ValueError("Quantidade de linhas de treino divergente.")
        buffer=StringIO()
        with redirect_stdout(buffer):
            selected.explain(mode="formatted")
        plans[str(partitions)]=buffer.getvalue()
        runs.append({"shuffle_partitions_requested":partitions,
                     "comparison_seconds":seconds, **comparison})
        print(json.dumps(runs[-1],ensure_ascii=False))
        print(plans[str(partitions)])
finally:
    spark.conf.set("spark.sql.shuffle.partitions",previous_partitions)

{"shuffle_partitions_requested": 2, "comparison_seconds": 3.6916320999999925, "rows_union": 7485, "rows_left": 7485, "rows_right": 7485, "mismatched_rows": 0, "max_diff_geracao_em_history_end": 0.0, "max_diff_media_historico_mwmed": 0.0, "max_diff_target_generation": 0.0}
== Physical Plan ==
AdaptiveSparkPlan (23)
+- == Initial Plan ==
   PhotonResultStage (22)
   +- PhotonColumnarToRow (21)
      +- PhotonProject (20)
         +- PhotonFilter (19)
            +- PhotonWindow (18)
               +- PhotonProject (17)
                  +- PhotonWindow (16)
                     +- PhotonSort (15)
                        +- PhotonProject (14)
                           +- PhotonBroadcastHashJoin LeftOuter (13)
                              :- PhotonGenerate (7)
                              :  +- PhotonAgg (6)
                              :     +- PhotonShuffleExchangeSource (5)
                              :        +- PhotonShuffleMapStage (4)
                              :           

## 6. Treino como tabela nomeada; versão fixa para consumo

Publicamos uma tabela de treino para poder consultá-la por SQL e registrar sua identidade. O recálculo abaixo usa novamente a versão explícita da fonte. O número da versão capturado é consultado no histórico, não presumido como 0.

No laboratório, esta tabela é uma entrega de dados para um futuro estimador; **nenhum modelo é treinado**. Se a tabela preexistir e tiver sido alterada, a comparação deve falhar, não sobrescrevê-la silenciosamente.

In [0]:
start=time.perf_counter()
base=spark.read.option("versionAsOf",source_info["version"]).table(SOURCE_TABLE)
features=construir_features(completar_horas(base),PARAMS["horizon_hours"],
    PARAMS["availability_delay_hours_assumed"],PARAMS["window_hours"])
treino=selecionar_treino(features,CORTE)
training_info=publicar_uma_vez(treino,TRAIN_TABLE,"training")
training_pinned=spark.read.option("versionAsOf",training_info["version"]).table(TRAIN_TABLE)
training_comparison=comparar_frames(training_pinned,referencia,["serie","forecast_origin"])
publication_seconds=time.perf_counter()-start
print("Treino fixado:",json.dumps(training_info,ensure_ascii=False))
query=f"""SELECT COUNT(*) AS linhas,
    MIN(forecast_origin) AS primeira_origem,
    MAX(forecast_origin) AS ultima_origem,
    MAX(target_available_at) AS ultimo_alvo_disponivel
FROM {TRAIN_TABLE} VERSION AS OF {training_info['version']}"""
print(query)
spark.sql(query).show(truncate=False)
spark.sql(f"DESCRIBE HISTORY {TRAIN_TABLE}").select("version","timestamp","operation").show(truncate=False)

Treino fixado: {"name": "`workspace`.`renewcast_m07`.`treino_m073_f66e094498ef`", "table_id": "f0f6f4fa-3879-4e07-8f5c-3c89b9f2e128", "version": 0, "reused": false, "format": "delta"}
SELECT COUNT(*) AS linhas,
    MIN(forecast_origin) AS primeira_origem,
    MAX(forecast_origin) AS ultima_origem,
    MAX(target_available_at) AS ultimo_alvo_disponivel
FROM `workspace`.`renewcast_m07`.`treino_m073_f66e094498ef` VERSION AS OF 0
+------+-------------------+-------------------+----------------------+
|linhas|primeira_origem    |ultima_origem      |ultimo_alvo_disponivel|
+------+-------------------+-------------------+----------------------+
|7485  |2025-09-01 07:00:00|2026-07-10 03:00:00|2026-07-11 05:00:00   |
+------+-------------------+-------------------+----------------------+

+-------+-------------------+----------------------+
|version|timestamp          |operation             |
+-------+-------------------+----------------------+
|0      |2026-10-29 23:25:37|CREATE TABLE AS SELEC

## 7. Recibo da execução e entrega

O recibo guarda os nomes, IDs e versões, parâmetros, hashes de entrada, versões de Python/Spark, durações e os dois planos. A versão local de M07.2 não é transportada como se fosse a versão cloud: são tabelas diferentes.

Exporte este notebook com as saídas em **File → Export → IPython notebook → Include outputs**. Baixe também o recibo pelo Catalog Explorer. O notebook exportado permite revisar o código que você realmente executou; o hash `distributed_notebook_sha256` abaixo identifica apenas o arquivo que foi fornecido originalmente.

In [0]:
for name,info in manifest["files"].items():
    if sha_file(INPUT_DIR/name) != info["sha256"]:
        raise ValueError("Entrada modificada durante a execucao.")
if spark.sql(f"DESCRIBE DETAIL {SOURCE_TABLE}").first()["id"] != source_info["table_id"]:
    raise ValueError("A tabela de origem foi substituida durante a execucao.")
if spark.sql(f"DESCRIBE DETAIL {TRAIN_TABLE}").first()["id"] != training_info["table_id"]:
    raise ValueError("A tabela de treino foi substituida durante a execucao.")
run_id=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"_"+uuid.uuid4().hex[:8]
receipt={
    "schema":"renewcast-databricks-receipt-v1", "completed":True, "run_id":run_id,
    "package_id":PACKAGE_ID, "input_manifest_sha256":sha_file(INPUT_DIR/"manifesto_entrada.json"),
    "input_files":manifest["files"], "distributed_notebook_sha256":manifest["distributed_notebook_sha256"],
    "executed_notebook_source_hash_verified":False,
    "implementation_label":IMPL_ID, "spark_version":spark.version, "python_version":platform.python_version(),
    "edition_declared":EDICAO, "compute_mode":"serverless", "cost_measured":False, "cost_amount":None,
    "parameters":PARAMS, "source_table":source_info, "training_table":training_info,
    "source_comparison":source_comparison, "configurations":runs,
    "training_comparison":training_comparison, "publish_and_verify_seconds":publication_seconds,
    "plans":plans, "availability_is_assumed":True, "historical_source_versions_verified":False,
    "models_trained":False, "performance_superiority_established":False,
}
output_dir=ROOT_VOLUME/"evidencias_m073"
output_dir.mkdir(exist_ok=True)
receipt_path=output_dir/f"recibo_{run_id}.json"
with receipt_path.open("x",encoding="utf-8") as handle:
    json.dump(receipt,handle,ensure_ascii=False,indent=2,allow_nan=False)
print("REPRODUCAO CONFERIDA: 8760 registros de origem; 7485 linhas de treino; zero divergencias nas duas configuracoes.")
print("Recibo:",receipt_path)
print("Proxima entrega: notebook exportado com saidas e reports/scale_comparison.md conforme a celula seguinte.")

REPRODUCAO CONFERIDA: 8760 registros de origem; 7485 linhas de treino; zero divergencias nas duas configuracoes.
Recibo: /Volumes/workspace/renewcast_m07/arquivos/evidencias_m073/recibo_20261029T232749Z_4848481e.json
Proxima entrega: notebook exportado com saidas e reports/scale_comparison.md conforme a celula seguinte.


## 8. Sua decisão de engenharia — uma entrega, quatro evidências

Escreva `reports/scale_comparison.md` no seu projeto, usando esta execução:

1. **Portabilidade:** registre o nome, o ID e a versão da tabela de origem e da tabela de treino. Explique por que usar “versão 0” sem dizer qual tabela é insuficiente. Diferencie reproduzir o dataset fixado de usar uma revisão posterior em um experimento novo. Isso consolida a interpretação de M07.2.
2. **Plano:** localize um `Exchange` que alimenta a janela e transcreva o trecho curto. Explique qual chave deve permanecer junta e por que repartir a janela por dia adulteraria o histórico.
3. **Medição:** apresente as durações de comparação das configurações 2 e 4, com zero divergências. Diga quais efeitos impedem interpretar essa amostra como superioridade de desempenho: aquecimento, ordem, otimização adaptativa e custo da própria auditoria.
4. **Decisão:** para este recorte de 8.760 horas, escolha onde manter a exploração cotidiana e quando o Databricks passaria a justificar a complexidade. Use necessidades concretas, como execução compartilhada, permissões e vários conjuntos; não “cloud é sempre mais rápido”.

Não é necessário refazer Delta, calendário ou treino. Entregue o notebook exportado, o recibo e esse parecer. Aceite técnico: correspondência integral, versões identificadas e ausência de reescrita silenciosa. Aceite de aprendizagem: defender a decisão com o plano e os números observados.